In [ ]:
!pip install -U diffusers transformers accelerate -q

In [1]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")

CUDA available: True
Allocated: 0.00 GB
Reserved:  0.00 GB


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("تم تسجيل الدخول ✅")

تم تسجيل الدخول ✅


In [7]:
import torch, gc
from diffusers import StableDiffusion3Pipeline

pipe = StableDiffusion3Pipeline.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    text_encoder_3=None,
    tokenizer_3=None,
    variant="fp16",
)
pipe = pipe.to("cuda")   # بدون فرض dtype هالمرة

print("✅ الموديل جاهز")

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 54.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 14.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 13.96 GiB is allocated by PyTorch, and 464.98 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [6]:
import requests
def brief_to_image_prompt(b):
    return f"{b['logo_direction']}, professional corporate branding, flat vector design, Adobe Illustrator style, geometric, minimalist icon, clean background, no text, no cartoon, no mascot"

briefs = requests.get("https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json").json()[:5]

generated_files = []
for brief in briefs:
    print(f"⏳ {brief['id']}...")
    image = pipe(
        prompt=brief_to_image_prompt(brief),
        num_inference_steps=28,
        guidance_scale=7.0,
        generator=torch.Generator("cuda").manual_seed(42),
    ).images[0]

    filename = f"SD3_{brief['id']}.png"
    image.save(filename)
    generated_files.append(filename)
    print(f"✅ {brief['id']}")

    del image
    gc.collect()
    torch.cuda.empty_cache()

print("✅ خلصت")

⏳ BR001...


  0%|          | 0/28 [00:00<?, ?it/s]

RuntimeError: Input type (c10::Half) and bias type (float) should be the same